In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Care_Load_Project/HHS_Unaccompanied_Alien_Children_Program.csv")

In [ ]:
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


In [ ]:
df.shape

(1170, 6)

In [ ]:
df.columns

Index(['Date', 'Children apprehended and placed in CBP custody*',
       'Children in CBP custody', 'Children transferred out of CBP custody',
       'Children in HHS Care', 'Children discharged from HHS Care'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1170 entries, 0 to 1169
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Date                                             720 non-null    object 
 1   Children apprehended and placed in CBP custody*  720 non-null    float64
 2   Children in CBP custody                          720 non-null    float64
 3   Children transferred out of CBP custody          720 non-null    float64
 4   Children in HHS Care                             720 non-null    object 
 5   Children discharged from HHS Care                720 non-null    float64
dtypes: float64(4), object(2)
memory usage: 55.0+ KB


# ***1.  Time Series Preparation***

In [ ]:
# convert Date type
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df.info()

In [ ]:
df = df.sort_values(by='Date')

In [ ]:
df.set_index('Date', inplace = True)

In [ ]:
df.head()

In [ ]:
type(df.index)

In [ ]:
#check if dates are daily or not in next 2 cells

df.index.min(), df.index.max()


In [ ]:
# df = df.asfreq('D')  error shows that date has duplicates values


In [ ]:
# check duplicates first
df.index.duplicated().sum()

In [ ]:
# combine those rows of same date or index

df = df.groupby(df.index).sum()

In [ ]:
# now we will apply daily date series

df = df.asfreq('D')

In [ ]:
df.head()
# some rows ahve NaN means misssing data

**Handling missing values using linear interplolation**

In [ ]:
df = df.interpolate(method='linear')

In [ ]:
df.head()
# done

**Time-Series Decomposition :**

In [ ]:
!pip install statsmodels

In [ ]:
df['Children in HHS Care'] = df['Children in HHS Care'].astype(str).str.replace(',', '', regex=True)
df['Children in HHS Care'] = pd.to_numeric(df['Children in HHS Care'], errors='coerce')

#  I've updated the code to use errors='coerce' in pd.to_numeric. This will convert any unparseable strings (like 'nan')
# into proper numerical NaN values, allowing the interpolate method to fill them correctly.
# Re-interpolate the dataframe to fill any NaNs that might exist in the newly numeric column.
# The previous interpolation might not have fully converted or handled NaNs in object-type columns correctly.
df = df.interpolate(method='linear')



In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
result = seasonal_decompose(df['Children in HHS Care'], model='additive', period=7)
fig = result.plot()
plt.show()

# ***2. Feature Engineering for Forecasting***

In [ ]:
# creating lag features

df['lag_1'] = df['Children in HHS Care'].shift(1)
df['lag_2'] = df['Children in HHS Care'].shift(7)
df['lag_3'] = df['Children in HHS Care'].shift(14)

df.head(15)

In [ ]:
# drop rows containing nan
df = df.dropna()

In [ ]:
df.shape  #remaining rows

In [ ]:
# creating rolling features

# rolling mean/average - to know direction over last 7 days and last 14 days means short-term trend
df['roll_mean_7'] = df['Children in HHS Care'].rolling(window = 7).mean()
df['roll_mean_14'] = df['Children in HHS Care'].rolling(window = 14).mean()

# rolling std - to know how unstable our system were over last 7 and 14 days
df['roll_std_7'] = df['Children in HHS Care'].rolling(window = 7).std()
df['roll_std_14'] = df['Children in HHS Care'].rolling(window = 14).std()

# drop rows having Nan
df = df.dropna()
df.head()

In [ ]:
# net pressure indicator feature
df['net_flow'] = (df['Children transferred out of CBP custody'] - df['Children discharged from HHS Care'])

In [ ]:
# Calender Effects

df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)


In [ ]:
!pip install holidays

In [ ]:
# holiday proxy
import holidays
us_holidays = holidays.US()
df['is_holiday'] = df.index.isin(us_holidays).astype(int)

In [ ]:
# extra cal signals
df['is_month_start'] = df.index.is_month_start.astype(int)
df['is_month_end'] = df.index.is_month_end.astype(int)

In [ ]:
df = df.dropna()
df.head(7)

# ***3. TRAIN-TEST STRATEGY***

**Strict Time-based splitting**

In [ ]:
# strict time-based split

# define features & target
x = df.drop(columns=['Children in HHS Care'])
y = df['Children in HHS Care']

# 80% trian, 20% test
split = int(len(df) * 0.8)

x_train = x.iloc[:split]
x_test = x.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)

y_pred =  model.predict(x_test)

mae_strict = mean_absolute_error(y_test, y_pred)
print("Strict Split MAE:", mae_strict)

**Walk-forward Validation splitting**

In [ ]:
# Walk-forward Validation - stimulates real world forecasting

predictions = []
actuals = []


for i in range(split, len(df)):
  model = RandomForestRegressor(n_estimators = 30, random_state=42)

  x_train_wf = x.iloc[:i]
  y_train_wf = y.iloc[:i]

  x_test_wf = x.iloc[i:i+1]
  y_test_wf = y.iloc[i:i+1]

  model.fit(x_train_wf, y_train_wf)

  pred = model.predict(x_test_wf)[0]

  predictions.append(pred)
  actuals.append(y_test_wf.values[0])


mae_wf = mean_absolute_error(actuals, predictions)
print("Walk Forward MAE:", mae_wf)

**Multi-Horizon Evaluation**

In [ ]:
df['target_1'] = df['Children in HHS Care'].shift(-1)
df['target_7'] = df['Children in HHS Care'].shift(-7)
df['target_14'] = df['Children in HHS Care'].shift(-14)

df = df.dropna()

*Evaluation :*

In [ ]:
features = [
    'lag_1', 'lag_2', 'lag_3',
    'roll_mean_7', 'roll_std_7',
    'roll_mean_14', 'roll_std_14',
    'net_flow',
    'day_of_week', 'month',
    'is_weekend', 'is_holiday',
    'is_month_start', 'is_month_end'
]

X = df[features]

In [ ]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]

In [ ]:
horizons = {
    "1_day": "target_1",
    "7_day": "target_7",
    "14_day": "target_14"
}

for name, target in horizons.items():

    y = df[target]

    y_train = y.iloc[:split]
    y_test  = y.iloc[split:]

    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)

    print(f"{name} Forecast MAE: {mae}")

**VISUALIZE PREDICTIONS :**

In [ ]:
# 1-day forecast only(for plotting)

target = 'target_1'

x = df[features]
y = df[target]

split = int(len(df) * 0.8)

x_train = x.iloc[:split]
x_test = x.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = RandomForestRegressor(n_estimators = 100, random_state = 42)
model.fit(x_train, y_train)

preds =  model.predict(x_test)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(y_test.values, label="Actual")
plt.plot(preds, label="Predicted")
plt.legend()
plt.title("1-Day Forecast")
plt.show()

***Conclusion ::***
The 1-day forecast model successfully captures overall trend dynamics. While short-term fluctuations are slightly smoothed, the model maintains stable predictive behavior with low average error (~67 units), indicating strong short-term forecasting capability.

In [ ]:
# 7-day forecast only (for plotting)

target = 'target_7'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds_7 = model.predict(X_test)

In [ ]:
plt.figure(figsize=(11,6))
plt.plot(y_test.values, label="Actual")
plt.plot(preds_7, label="Predicted")
plt.legend()
plt.title("7-Day Forecast")
plt.show()

***Conclusion ::***
The 7-day forecast maintains trend alignment but exhibits increased smoothing and lag at turning points compared to the 1-day model. This behavior is expected due to higher uncertainty in medium-term predictions, reflected in the increased MAE (~92).

In [ ]:
# 14-day forecast only (for plotting)

target = 'target_14'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds_14 = model.predict(X_test)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(y_test.values, label="Actual")
plt.plot(preds_14, label="Predicted")
plt.legend()
plt.title("14-Day Forecast")
plt.show()

***Conclusion ::***
The 14-day forecast shows that the model successfully captures the overall trend of care load changes but struggles to accurately predict sharp fluctuations.

Key observations:

• The predicted line is smoother than the actual values
• The model underestimates sudden drops
• Recovery trends are captured but with noticeable lag
• Error is visibly larger compared to 1-day and 7-day forecasts

This behavior is expected in long-horizon forecasting because:

Uncertainty increases as the prediction window expands

The model relies more on historical patterns

Sudden shocks are difficult to anticipate 14 days in advance

# ***4. FORECASTING MODELS***

# **Baseline Models**

1.   naive persistence model
2.   Moving average forecast



In [ ]:
# # naive 1day ahead forecast using strict split

# # predictions = previous day's actual values
# # ex prdection for day 2 = day 1 value
# y_pred_naive = y_test.shift(1)

# # remove the first value that becomes NaN after shifting
# y_pred_naive = y_pred_naive.dropna()

# # ensures y_actual & y_pred_naive have same dates
# y_actual = y_test.loc[y_pred_naive.index]

# # import mae func
# from sklearn.metrics import mean_absolute_error

# # calculate mae
# mae_naive_1 = mean_absolute_error(y_actual, y_pred_naive)
# rmse_naive_1 = np.sqrt(mean_squared_error(y_actual, y_pred_naive))
# mape_naive_1 = np.mean(
#     np.abs((np.array(y_actual) - np.array(y_pred_naive))
#            / np.array(y_actual))
# ) * 100

# print("Naive 1_Day MAE :", mae_naive_1)
# print("Naive 1_Day RMSE :", rmse_naive_1)
# print("Naive 1_Day MAPE:", mape_naive_1)

In [ ]:
# naive 1day ahead forecast using walk-forward
from sklearn.metrics import mean_absolute_error

# convert training data to list - to keep adding new values 1 by 1
history = list(y_train)

# empty list of predictions
predictions = []

# start walk-frwd loop
for t in range(len(y_test)):
  yhat = history[-1]         # tommorow = today
  predictions.append(yhat)
  history.append(y_test.iloc[t]) #adding the actual real value to taining set

mae_naive_wf_1 = mean_absolute_error(y_test, predictions)
rmse_naive_wf_1 = np.sqrt(mean_squared_error(y_test, predictions))
mape_naive_wf_1 = np.mean(
    np.abs((np.array(y_test) - np.array(predictions))
           / np.array(y_test))
) * 100

print("Naive Walk-Forward 1-day MAE :", mae_naive_wf_1)
print("Naive Walk-Forward 1-day RMSE :", rmse_naive_wf_1)
print("Naive Walk-Forward 1-day MAPE :", mape_naive_wf_1)



In [ ]:
# naive 7day ahead forecast using walk-forward
history = list(y_train)
predictions_7 = []

for t in range(len(y_test)):

  if len(history) >= 7:
    yhat = history[-7]   #value of 7day ago
  else:
    yhat = history[-1]

  predictions_7.append(yhat)
  history.append(y_test.iloc[t])

mae_naive_wf_7 = mean_absolute_error(y_test, predictions_7)
rmse_naive_wf_7 = np.sqrt(mean_squared_error(y_test, predictions_7))
mape_naive_wf_7 = np.mean(
    np.abs((np.array(y_test) - np.array(predictions_7))
           / np.array(y_test))
) * 100

print("Naive Walk-Forward 7-day MAE :", mae_naive_wf_7)
print("Naive Walk-Forward 7-day RMSE :", rmse_naive_wf_7)
print("Naive Walk-Forward 7-day MAPE :", mape_naive_wf_7)


In [ ]:
# naive 7day ahead forecast using walk-forward
history = list(y_train)
predictions_14 = []

for t in range(len(y_test)):

  if len(history) >= 14:
    yhat = history[-14]   #value of 14day ago
  else:
    yhat = history[-1]

  predictions_14.append(yhat)
  history.append(y_test.iloc[t])

mae_naive_wf_14 = mean_absolute_error(y_test, predictions_14)
rmse_naive_wf_14 = np.sqrt(mean_squared_error(y_test, predictions_14))
mape_naive_wf_14 = np.mean(
    np.abs((np.array(y_test) - np.array(predictions_14))
           / np.array(y_test))
) * 100
print("Naive Walk-Forward 14-day MAE :", mae_naive_wf_14)
print("Naive Walk-Forward 14-day RMSE :", rmse_naive_wf_14)
print("Naive Walk-Forward 14-day MAPE :", mape_naive_wf_14)



In [ ]:
# ===============================
# NAIVE HORIZON METRICS
# ===============================

naive_horizon_metrics = {
    "1-Day": {
        "MAE": mae_naive_wf_1,
        "RMSE": rmse_naive_wf_1,
        "MAPE": mape_naive_wf_1
    },
    "7-Day": {
        "MAE": mae_naive_wf_7,
        "RMSE": rmse_naive_wf_7,
        "MAPE": mape_naive_wf_7
    },
    "14-Day": {
        "MAE": mae_naive_wf_14,
        "RMSE": rmse_naive_wf_14,
        "MAPE": mape_naive_wf_14
    }
}

print("\nNaive Horizon Metrics:")
for h, m in naive_horizon_metrics.items():
    print(h, ":", m)

*con: *

| Horizon | Naïve Walk-Forward MAE |
| ------- | ---------------------- |
| 1-Day   | ~6.67                  |
| 7-Day   | ~37.27                 |
| 14-Day  | ~71.54                 |

Baseline naïve forecasts show strong short-term predictability (MAE ≈ 6.7) with error increasing progressively for 7-day (≈37.3) and 14-day (≈71.5) horizons, reflecting growing forecast uncertainty over longer intervals.


In [ ]:
# moving avg 1-day forcast using walk-frwd
history = list(y_train)
predictions_ma = []
window = 7   #7-day moving average

for t in range(len(y_test)):

  if len(history) >= window:
    yhat = np.mean(history[-window:])
  else:
    yhat = np.mean(history)

  predictions_ma.append(yhat)
  history.append(y_test.iloc[t])

mae_ma_1 = mean_absolute_error(y_test, predictions_ma)
rmse_ma_1 = np.sqrt(mean_squared_error(y_test, predictions_ma))
mape_ma_1 = np.mean(
    np.abs((np.array(y_test) - np.array(predictions_ma))
           / np.array(y_test))
) * 100


print("Moving Average 1-day MAE :", mae_ma_1)
print("Moving Average 1-day RMSE :", rmse_ma_1)
print("Moving Average 1-day MAPE :", mape_ma_1)

In [ ]:
# moving avg 7-day forecast using walk-forward
history = list(y_train)
predictions_ma_7 = []
window = 7   # 7-day moving average

for t in range(len(y_test) - 6):   # -6 because 7-step ahead

  if len(history) >= window:
    yhat = np.mean(history[-window:])
  else:
    yhat = np.mean(history)

  predictions_ma_7.append(yhat)
  history.append(y_test.iloc[t])

# Align actual values (shift by 6)
actual_7 = y_test.iloc[6:]

mae_ma_7 = mean_absolute_error(actual_7, predictions_ma_7)
rmse_ma_7 = np.sqrt(mean_squared_error(actual_7, predictions_ma_7))
mape_ma_7 = np.mean(
    np.abs((np.array(actual_7) - np.array(predictions_ma_7))
           / np.array(actual_7))
) * 100

print("Moving Average 7-day MAE :", mae_ma_7)
print("Moving Average 7-day RMSE :", rmse_ma_7)
print("Moving Average 7-day MAPE :", mape_ma_7)

In [ ]:
# moving avg 14-day forecast using walk-forward
history = list(y_train)
predictions_ma_14 = []
window = 14   # 14-day moving average

for t in range(len(y_test) - 13):   # -13 because 14-step ahead

  if len(history) >= window:
    yhat = np.mean(history[-window:])
  else:
    yhat = np.mean(history)

  predictions_ma_14.append(yhat)
  history.append(y_test.iloc[t])

# Align actual values (shift by 13)
actual_14 = y_test.iloc[13:]

mae_ma_14 = mean_absolute_error(actual_14, predictions_ma_14)
rmse_ma_14 = np.sqrt(mean_squared_error(actual_14, predictions_ma_14))
mape_ma_14 = np.mean(
    np.abs((np.array(actual_14) - np.array(predictions_ma_14))
           / np.array(actual_14))
) * 100

print("Moving Average 14-day MAE :", mae_ma_14)
print("Moving Average 14-day RMSE :", rmse_ma_14)
print("Moving Average 14-day MAPE :", mape_ma_14)

In [ ]:
# ===============================
# MOVING AVERAGE HORIZON METRICS
# ===============================

ma_horizon_metrics = {
    "1-Day": {
        "MAE": mae_ma_1,
        "RMSE": rmse_ma_1,
        "MAPE": mape_ma_1
    },
    "7-Day": {
        "MAE": mae_ma_7,
        "RMSE": rmse_ma_7,
        "MAPE": mape_ma_7
    },
    "14-Day": {
        "MAE": mae_ma_14,
        "RMSE": rmse_ma_14,
        "MAPE": mape_ma_14
    }
}

print("\nMoving Average Horizon Metrics:")
for h, m in ma_horizon_metrics.items():
    print(h, ":", m)

**con ::**

The Naïve model performed best among baseline methods for 1-day ahead forecasting with an MAE of 6.67. The Moving Average model produced a significantly higher error (MAE ≈ 21.48), indicating that smoothing reduced short-term prediction accuracy. Forecast error increased as the prediction horizon expanded (7-day MAE ≈ 37.27; 14-day MAE ≈ 71.54), demonstrating increasing uncertainty over longer horizons. Therefore, the Naïve persistence model was selected as the primary baseline benchmark.

# **Statistical Models**


1.   ARIMA / SARIMA (trend & seasonality)
2.  Exponential smoothing



In [ ]:
# arima 1-day forcast using walk-frwd

# import ARIMA
from statsmodels.tsa.arima.model import ARIMA
history = list(y_train)
predictions_arima = []

# start walk-frwd loop
for t in range(len(y_test)):

  # create arima model
  model = ARIMA(history, order=(1,1,1))
  # fit the model - this trains the model on current history
  model_fit = model.fit()

  #forcast next day,steps=1 → one day forecast
  yhat = model_fit.forecast(steps=1)[0]

  predictions_arima.append(yhat)
  history.append(y_test.iloc[t])

mae_arima_1 = mean_absolute_error(y_test, predictions_arima)
rmse_arima_1 = np.sqrt(mean_squared_error(y_test, predictions_arima))
mape_arima_1  = np.mean(np.abs((np.array(y_test) - np.array(predictions_arima)) /np.array(y_test))) * 100


print("ARIMA (1,1,1) 1-day walk-frwd MAE :", mae_arima_1)
print("ARIMA (1,1,1) 1-day walk-frwd RMSE :", rmse_arima_1)
print("ARIMA (1,1,1) 1-day walk-frwd MAPE :", mape_arima_1)

In [ ]:
# arima 7-day forcast using walk-frwd

history = list(y_train)
predictions_arima_7 = []

horizon = 7

for t in range(len(y_test) - horizon + 1):

    model = ARIMA(history, order=(1,1,1))
    model_fit = model.fit()

    forecast = model_fit.forecast(steps=horizon)
    yhat = forecast[-1]   # 7th day prediction

    predictions_arima_7.append(yhat)

    history.append(y_test.iloc[t])

actual_7 = y_test[horizon-1:]     #This means: Start comparing from the 7th real value,Because
                                    # first 6 actual values don’t have matching predictions.

mae_arima_7 = mean_absolute_error(actual_7, predictions_arima_7)

rmse_arima_7 = np.sqrt(
    mean_squared_error(actual_7, predictions_arima_7)
)

mape_arima_7 = np.mean(
    np.abs(
        (np.array(actual_7) - np.array(predictions_arima_7))
        / np.array(actual_7)
    )
) * 100

# ---- PRINT RESULTS ----

print("ARIMA 7-Day Walk-Forward MAE :", mae_arima_7)
print("ARIMA 7-Day Walk-Forward RMSE:", rmse_arima_7)
print("ARIMA 7-Day Walk-Forward MAPE:", mape_arima_7)


In [ ]:
# ARIMA 14-day forecast using walk-forward

history = list(y_train)
predictions_arima_14 = []

horizon = 14

for t in range(len(y_test) - horizon + 1):

    model = ARIMA(history, order=(1,1,1))
    model_fit = model.fit()

    forecast = model_fit.forecast(steps=horizon)
    yhat = forecast[-1]   # 14th day prediction

    predictions_arima_14.append(yhat)

    history.append(y_test.iloc[t])

# Align actual values
actual_14 = y_test[horizon-1:]

# ---- METRICS ----

mae_arima_14 = mean_absolute_error(actual_14, predictions_arima_14)

rmse_arima_14 = np.sqrt(
    mean_squared_error(actual_14, predictions_arima_14)
)

mape_arima_14 = np.mean(
    np.abs(
        (np.array(actual_14) - np.array(predictions_arima_14))
        / np.array(actual_14)
    )
) * 100

print("ARIMA 14-Day Walk-Forward MAE :", mae_arima_14)
print("ARIMA 14-Day Walk-Forward RMSE:", rmse_arima_14)
print("ARIMA 14-Day Walk-Forward MAPE:", mape_arima_14)

In [ ]:
arima_results ={}
arima_results = {
    "mae": [mae_arima_1, mae_arima_7, mae_arima_14],
    "rmse": [rmse_arima_1, rmse_arima_7, rmse_arima_14],
    "mape": [mape_arima_1, mape_arima_7, mape_arima_14]
}
print(arima_results)

In [ ]:
# ===============================
# ARIMA HORIZON ERROR METRICS
# ===============================

arima_horizon_metrics = {
    "1-Day": {
        "MAE": mae_arima_1,
        "RMSE": rmse_arima_1,
        "MAPE": mape_arima_1
    },
    "7-Day": {
        "MAE": mae_arima_7,
        "RMSE": rmse_arima_7,
        "MAPE": mape_arima_7
    },
    "14-Day": {
        "MAE": mae_arima_14,
        "RMSE": rmse_arima_14,
        "MAPE": mape_arima_14
    }
}

print("\nARIMA Horizon Metrics:")
for h, m in arima_horizon_metrics.items():
    print(h, ":", m)

**con ::**

Multi-horizon forecasting performance was evaluated using walk-forward validation. The ARIMA (1,1,1) model consistently outperformed the Naïve persistence baseline across all forecast horizons. For 1-day forecasting, ARIMA achieved an MAE of approximately 4.99 compared to 6.67 for the Naïve model. Similar improvements were observed for 7-day (32.28 vs 37.27) and 14-day horizons (67.44 vs 71.54).

These results indicate that the time series contains underlying temporal structure beyond simple persistence, which ARIMA successfully captures through autoregressive and differencing components. As expected, forecast error increased with longer horizons due to growing uncertainty.

In [ ]:
# ETS 1-Day Walk-Forward

from statsmodels.tsa.holtwinters import ExponentialSmoothing

history = list(y_train)
predictions_ets_1 = []

for t in range(len(y_test)):

  model = ExponentialSmoothing(history, trend='add', seasonal=None)
  model_fit = model.fit()

  yhat = model_fit.forecast(1)[0]
  predictions_ets_1.append(yhat)

  history.append(y_test.iloc[t])

# ---- METRICS ----

mae_ets_1 = mean_absolute_error(y_test, predictions_ets_1)

rmse_ets_1 = np.sqrt(
    mean_squared_error(y_test, predictions_ets_1)
)

mape_ets_1 = np.mean(
    np.abs(
        (np.array(y_test) - np.array(predictions_ets_1))
        / np.array(y_test)
    )
) * 100

print("ETS 1-day MAE :", mae_ets_1)
print("ETS 1-day RMSE:", rmse_ets_1)
print("ETS 1-day MAPE:", mape_ets_1)

In [ ]:
# ETS 7-Day Walk-Forward

history = list(y_train)
predictions_ets_7 = []

horizon = 7

for t in range(len(y_test) - horizon + 1):

  model = ExponentialSmoothing(history, trend='add', seasonal=None)
  model_fit = model.fit()

  forecast = model_fit.forecast(horizon)
  yhat = forecast[-1]

  predictions_ets_7.append(yhat)

  history.append(y_test.iloc[t])

actual_7 = y_test[horizon-1:]

mae_ets_7 = mean_absolute_error(actual_7, predictions_ets_7)

rmse_ets_7 = np.sqrt(
    mean_squared_error(actual_7, predictions_ets_7)
)

mape_ets_7 = np.mean(
    np.abs(
        (np.array(actual_7) - np.array(predictions_ets_7))
        / np.array(actual_7)
    )
) * 100

print("ETS 7-day MAE :", mae_ets_7)
print("ETS 7-day RMSE:", rmse_ets_7)
print("ETS 7-day MAPE:", mape_ets_7)

In [ ]:
# ETS 14-Day Walk-Forward

history = list(y_train)
predictions_ets_14 = []

horizon = 14

for t in range(len(y_test) - horizon + 1):

  model = ExponentialSmoothing(history, trend='add', seasonal=None)
  model_fit = model.fit()

  forecast = model_fit.forecast(horizon)
  yhat = forecast[-1]

  predictions_ets_14.append(yhat)

  history.append(y_test.iloc[t])

actual_14 = y_test[horizon-1:]

mae_ets_14 = mean_absolute_error(actual_14, predictions_ets_14)

rmse_ets_14 = np.sqrt(
    mean_squared_error(actual_14, predictions_ets_14)
)

mape_ets_14 = np.mean(
    np.abs(
        (np.array(actual_14) - np.array(predictions_ets_14))
        / np.array(actual_14)
    )
) * 100

print("ETS 14-day MAE :", mae_ets_14)
print("ETS 14-day RMSE:", rmse_ets_14)
print("ETS 14-day MAPE:", mape_ets_14)

In [ ]:
ets_results ={}
ets_results = {
    "mae": [mae_ets_1, mae_ets_7, mae_ets_14],
    "rmse": [rmse_ets_1, rmse_ets_7, rmse_ets_14],
    "mape": [mape_ets_1, mape_ets_7, mape_ets_14]
}
print(ets_results)

In [ ]:
# ===============================
# ETS HORIZON ERROR METRICS
# ===============================

ets_horizon_metrics = {
    "1-Day": {
        "MAE": mae_ets_1,
        "RMSE": rmse_ets_1,
        "MAPE": mape_ets_1
    },
    "7-Day": {
        "MAE": mae_ets_7,
        "RMSE": rmse_ets_7,
        "MAPE": mape_ets_7
    },
    "14-Day": {
        "MAE": mae_ets_14,
        "RMSE": rmse_ets_14,
        "MAPE": mape_ets_14
    }
}

print("\nETS Horizon Metrics:")
for h, m in ets_horizon_metrics.items():
    print(h, ":", m)

**con ::** Exponential Smoothing outperforms both ARIMA and Naive

# **Machine Learning Models:**

1.   Random Forest Regressor
2.   Gradient Boosting Regressor



In [ ]:
# 1-day
# Target
target = 'target_1'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

predictions = []
actuals = []

for i in range(split, len(df)):

    X_train = X.iloc[:i]
    y_train = y.iloc[:i]

    X_test = X.iloc[i:i+1]
    y_test = y.iloc[i]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    predictions.append(pred)
    actuals.append(y_test)

rf_mae_1 = mean_absolute_error(actuals, predictions)
rf_rmse_1 = np.sqrt(mean_squared_error(actuals, predictions))
rf_mape_1 = np.mean(np.abs((np.array(actuals) - np.array(predictions)) /np.array(actuals))) * 100

print("Random Forest 1-Day MAE:", rf_mae_1)
print("Random Forest 1-Day RMSE:",rf_rmse_1)
print("Random Forest 1-Day MAPE:",rf_mape_1)


In [ ]:
# 7-day rf
target = 'target_7'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

predictions = []
actuals = []

for i in range(split, len(df)):

    X_train = X.iloc[:i]
    y_train = y.iloc[:i]

    X_test = X.iloc[i:i+1]
    y_test = y.iloc[i]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    predictions.append(pred)
    actuals.append(y_test)

rf_mae_7 = mean_absolute_error(actuals, predictions)
rf_rmse_7  = np.sqrt(mean_squared_error(actuals, predictions))
rf_mape_7 = np.mean(np.abs((np.array(actuals) - np.array(predictions)) /np.array(actuals))) * 100


print("Random Forest 7-Day MAE:", rf_mae_7)
print("Random Forest 7-Day RMSE:",rf_rmse_7)
print("Random Forest 7-Day MAPE:",rf_mape_7)


Random Forest 7-Day MAE: 23.72819093627791
Random Forest 7-Day RMSE: 35.213194096986435
Random Forest 7-Day MAPE: 1.0849244483693512


In [ ]:
# 14-day rf

target = 'target_14'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

predictions = []
actuals = []

for i in range(split, len(df)):

    X_train = X.iloc[:i]
    y_train = y.iloc[:i]

    X_test = X.iloc[i:i+1]
    y_test = y.iloc[i]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    predictions.append(pred)
    actuals.append(y_test)

rf_mae_14 = mean_absolute_error(actuals, predictions)
rf_rmse_14  = np.sqrt(mean_squared_error(actuals, predictions))
rf_mape_14 = np.mean(np.abs((np.array(actuals) - np.array(predictions)) /np.array(actuals))) * 100


print("Random Forest 14-Day MAE:", rf_mae_14)
print("Random Forest 14-Day RMSE:",rf_rmse_14)
print("Random Forest 14-Day MAPE:",rf_mape_14)

Random Forest 14-Day MAE: 25.491191281343454
Random Forest 14-Day RMSE: 40.37606636984854
Random Forest 14-Day MAPE: 1.1716958874625767


In [ ]:
# 1-day gradient
target = 'target_1'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

predictions = []
actuals = []

for i in range(split, len(df)):

    X_train = X.iloc[:i]
    y_train = y.iloc[:i]

    X_test = X.iloc[i:i+1]
    y_test = y.iloc[i]

    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    predictions.append(pred)
    actuals.append(y_test)

gb_mae_1 = mean_absolute_error(actuals, predictions)
gb_rmse_1 = np.sqrt(mean_squared_error(actuals, predictions))
gb_mape_1 = np.mean(np.abs((np.array(actuals) - np.array(predictions)) /np.array(actuals))) * 100


print("Gradient Boosting 1-Day MAE:", gb_mae_1)
print("Gradient Boosting 1-Day RMSE:", gb_rmse_1)
print("Gradient Boosting 1-Day MAPE:", gb_mape_1)

Gradient Boosting 1-Day MAE: 21.833516158005803
Gradient Boosting 1-Day RMSE: 27.677899644552333
Gradient Boosting 1-Day MAPE: 0.9926892459212123


In [ ]:
# 7-day gradient
target = 'target_7'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

predictions = []
actuals = []

for i in range(split, len(df)):

    X_train = X.iloc[:i]
    y_train = y.iloc[:i]

    X_test = X.iloc[i:i+1]
    y_test = y.iloc[i]

    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    predictions.append(pred)
    actuals.append(y_test)

gb_mae_7 = mean_absolute_error(actuals, predictions)
gb_rmse_7 = np.sqrt(mean_squared_error(actuals, predictions))
gb_mape_7 = np.mean(np.abs((np.array(actuals) - np.array(predictions)) /np.array(actuals))) * 100


print("Gradient Boosting 7-Day MAE:", gb_mae_7) #54.2
print("Gradient Boosting 7-Day RMSE:", gb_rmse_7)
print("Gradient Boosting 7-Day MAPE:", gb_mape_7)


Gradient Boosting 7-Day MAE: 54.216979397923545
Gradient Boosting 7-Day RMSE: 64.04387282888723
Gradient Boosting 7-Day MAPE: 2.453472242505066


In [ ]:
# 14-day gradient
target = 'target_14'

X = df[features]
y = df[target]

split = int(len(df) * 0.8)

predictions = []
actuals = []

for i in range(split, len(df)):

    X_train = X.iloc[:i]
    y_train = y.iloc[:i]

    X_test = X.iloc[i:i+1]
    y_test = y.iloc[i]

    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    predictions.append(pred)
    actuals.append(y_test)

gb_mae_14 = mean_absolute_error(actuals, predictions)
gb_rmse_14 = np.sqrt(mean_squared_error(actuals, predictions))
gb_mape_14 = np.mean(np.abs((np.array(actuals) - np.array(predictions)) /np.array(actuals))) * 100


print("Gradient Boosting 14-Day MAE:", gb_mae_14) #84.6
print("Gradient Boosting 14-Day RMSE:", gb_rmse_14)
print("Gradient Boosting 14-Day MAPE:", gb_mape_14)



Gradient Boosting 14-Day MAE: 84.63731463515846
Gradient Boosting 14-Day RMSE: 102.57916570379041
Gradient Boosting 14-Day MAPE: 3.7926770465467654


In [ ]:
rf_results = {}

rf_results['mae'] = [float(rf_mae_1), float(rf_mae_7), float(rf_mae_14)]
rf_results['rmse'] = [float(rf_rmse_1), float(rf_rmse_7), float(rf_rmse_14)]
rf_results['mape'] = [float(rf_mape_1), float(rf_mape_7), float(rf_mape_14)]

print(rf_results)

{'mae': [10.718626178974013, 23.72819093627791, 25.491191281343454], 'rmse': [13.518106166473894, 35.213194096986435, 40.37606636984854], 'mape': [0.47952357428577197, 1.0849244483693512, 1.1716958874625767]}


In [ ]:
# ===============================
# RANDOM FOREST HORIZON METRICS
# ===============================

rf_horizon_metrics = {
    "1-Day": {
        "MAE": float(rf_mae_1),
        "RMSE": float(rf_rmse_1),
        "MAPE": float(rf_mape_1)
    },
    "7-Day": {
        "MAE": float(rf_mae_7),
        "RMSE": float(rf_rmse_7),
        "MAPE": float(rf_mape_7)
    },
    "14-Day": {
        "MAE": float(rf_mae_14),
        "RMSE": float(rf_rmse_14),
        "MAPE": float(rf_mape_14)
    }
}

print("\nRandom Forest Horizon Metrics:")
for h, m in rf_horizon_metrics.items():
    print(h, ":", m)


Random Forest Horizon Metrics:
1-Day : {'MAE': 10.718626178974013, 'RMSE': 13.518106166473894, 'MAPE': 0.47952357428577197}
7-Day : {'MAE': 23.72819093627791, 'RMSE': 35.213194096986435, 'MAPE': 1.0849244483693512}
14-Day : {'MAE': 25.491191281343454, 'RMSE': 40.37606636984854, 'MAPE': 1.1716958874625767}


In [ ]:
gb_results ={}

gb_results['mae'] = [float(gb_mae_1), float(gb_mae_7), float(gb_mae_14)]
gb_results['rmse'] = [float(gb_rmse_1), float(gb_rmse_7), float(gb_rmse_14)]
gb_results['mape'] = [float(gb_mape_1), float(gb_mape_7), float(gb_mape_14)]

print(gb_results)

{'mae': [21.833516158005803, 54.216979397923545, 84.63731463515846], 'rmse': [27.677899644552333, 64.04387282888723, 102.57916570379041], 'mape': [0.9926892459212123, 2.453472242505066, 3.7926770465467654]}


In [ ]:
# ===============================
# GRADIENT BOOSTING HORIZON METRICS
# ===============================

gb_horizon_metrics = {
    "1-Day": {
        "MAE": float(gb_mae_1),
        "RMSE": float(gb_rmse_1),
        "MAPE": float(gb_mape_1)
    },
    "7-Day": {
        "MAE": float(gb_mae_7),
        "RMSE": float(gb_rmse_7),
        "MAPE": float(gb_mape_7)
    },
    "14-Day": {
        "MAE": float(gb_mae_14),
        "RMSE": float(gb_rmse_14),
        "MAPE": float(gb_mape_14)
    }
}

print("\nGradient Boosting Horizon Metrics:")
for h, m in gb_horizon_metrics.items():
    print(h, ":", m)


Gradient Boosting Horizon Metrics:
1-Day : {'MAE': 21.833516158005803, 'RMSE': 27.677899644552333, 'MAPE': 0.9926892459212123}
7-Day : {'MAE': 54.216979397923545, 'RMSE': 64.04387282888723, 'MAPE': 2.453472242505066}
14-Day : {'MAE': 84.63731463515846, 'RMSE': 102.57916570379041, 'MAPE': 3.7926770465467654}


**con ::**

Random Forest outperformed Gradient Boosting across all forecasting horizons. While both models showed increasing error with longer horizons, Gradient Boosting exhibited significant performance deterioration, particularly for the 14-day forecast. Random Forest demonstrated greater stability and better generalization ability in walk-forward validation, making it more suitable for multi-horizon forecasting in this dataset.

***Interpretations :::***


1.   Short-term forecasting is dominated by statistical models
2.  Random Forest handles nonlinear relationships better than classical models.
3.  Random Forest clearly outperforms all statistical models in long horizon.


# ***CONclusion  ***

The comparative analysis using walk-forward validation shows that statistical models (ETS and ARIMA) perform best for short-term (1-day) forecasting, indicating strong short-range temporal dependencies. However, as the forecast horizon increases, machine learning models—particularly Random Forest—demonstrate superior performance and stability. Random Forest achieved the lowest MAE for both 7-day and 14-day horizons, suggesting its ability to capture nonlinear patterns and complex feature interactions. Gradient Boosting exhibited significant performance degradation at longer horizons, indicating possible overfitting or sensitivity to noise in sequential forecasting tasks.

# ***5. Model Evaluation***

In [ ]:
import pandas as pd

comparison_data = [

    # ================= NAIVE =================
    ["Naive", "1-Day", mae_naive_wf_1, rmse_naive_wf_1, mape_naive_wf_1],
    ["Naive", "7-Day", mae_naive_wf_7, rmse_naive_wf_7, mape_naive_wf_7],
    ["Naive", "14-Day", mae_naive_wf_14, rmse_naive_wf_14, mape_naive_wf_14],

    # ================= MOVING AVERAGE =================
    ["Moving Average", "1-Day", mae_ma_1, rmse_ma_1, mape_ma_1],
    ["Moving Average", "7-Day", mae_ma_7, rmse_ma_7, mape_ma_7],
    ["Moving Average", "14-Day", mae_ma_14, rmse_ma_14, mape_ma_14],

    # ================= ARIMA =================
    ["ARIMA", "1-Day", mae_arima_1, rmse_arima_1, mape_arima_1],
    ["ARIMA", "7-Day", mae_arima_7, rmse_arima_7, mape_arima_7],
    ["ARIMA", "14-Day", mae_arima_14, rmse_arima_14, mape_arima_14],

    # ================= ETS =================
    ["ETS", "1-Day", mae_ets_1, rmse_ets_1, mape_ets_1],
    ["ETS", "7-Day", mae_ets_7, rmse_ets_7, mape_ets_7],
    ["ETS", "14-Day", mae_ets_14, rmse_ets_14, mape_ets_14],

    # ================= RANDOM FOREST =================
    ["Random Forest", "1-Day", rf_mae_1, rf_rmse_1, rf_mape_1],
    ["Random Forest", "7-Day", rf_mae_7, rf_rmse_7, rf_mape_7],
    ["Random Forest", "14-Day", rf_mae_14, rf_rmse_14, rf_mape_14],

    # ================= GRADIENT BOOSTING =================
    ["Gradient Boosting", "1-Day", gb_mae_1, gb_rmse_1, gb_mape_1],
    ["Gradient Boosting", "7-Day", gb_mae_7, gb_rmse_7, gb_mape_7],
    ["Gradient Boosting", "14-Day", gb_mae_14, gb_rmse_14, gb_mape_14],
]

comparison_df = pd.DataFrame(
    comparison_data,
    columns=["Model", "Horizon", "Horizon_Error(MAE)", "RMSE", "MAPE"]
)

# Round for clean presentation
comparison_df[["Horizon_Error(MAE)", "RMSE", "MAPE"]] = \
    comparison_df[["Horizon_Error(MAE)", "RMSE", "MAPE"]].round(3)

print(comparison_df.to_string(index=False))



            Model Horizon  Horizon_Error(MAE)    RMSE  MAPE
            Naive   1-Day               6.671   8.306 0.297
            Naive   7-Day              37.270  45.795 1.672
            Naive  14-Day              71.539  87.567 3.226
   Moving Average   1-Day              21.483  26.630 0.962
   Moving Average   7-Day              51.556  63.752 2.328
   Moving Average  14-Day             104.851 124.823 4.772
            ARIMA   1-Day               4.998   6.592 0.222
            ARIMA   7-Day              32.279  40.234 1.448
            ARIMA  14-Day              67.438  82.780 3.053
              ETS   1-Day               4.745   6.761 0.211
              ETS   7-Day              31.232  39.283 1.390
              ETS  14-Day              63.333  79.948 2.830
    Random Forest   1-Day              10.719  13.518 0.480
    Random Forest   7-Day              23.728  35.213 1.085
    Random Forest  14-Day              25.491  40.376 1.172
Gradient Boosting   1-Day              2

In [ ]:
# Sort by Best Model per Horizon

comparison_df.sort_values(["Horizon", "Horizon_Error(MAE)"])

,Model,Horizon,Horizon_Error(MAE),RMSE,MAPE
9,ETS,1-Day,4.745,6.761,0.211
6,ARIMA,1-Day,4.998,6.592,0.222
0,Naive,1-Day,6.671,8.306,0.297
12,Random Forest,1-Day,10.719,13.518,0.480
3,Moving Average,1-Day,21.483,26.630,0.962
15,Gradient Boosting,1-Day,21.834,27.678,0.993
14,Random Forest,14-Day,25.491,40.376,1.172
11,ETS,14-Day,63.333,79.948,2.830
8,ARIMA,14-Day,67.438,82.780,3.053
2,Naive,14-Day,71.539,87.567,3.226


**The evaluation demonstrates that statistical models such as ETS are highly effective for short-term forecasting due to their ability to capture trend dynamics. However, for medium and long-term horizons, Random Forest exhibits superior predictive performance and robustness, making it the preferred model for strategic forecasting applications.**

# ***6.  Key Performance Indicators (KPIs)***

**Based on multi-horizon MAE ranking, ETS and ARIMA consistently ranked among the top three models across all horizons, while Random Forest dominated medium and long-term forecasting. Therefore, these three models were selected for advanced KPI evaluation.**

In [ ]:
# Add Forecast Accuracy for each model & horizon

ets_accuracy_1 = 100 - mape_ets_1
ets_accuracy_7 = 100 - mape_ets_7
ets_accuracy_14 = 100 - mape_ets_14

arima_accuracy_1 = 100 - mape_arima_1
arima_accuracy_7 = 100 - mape_arima_7
arima_accuracy_14 = 100 - mape_arima_14

rf_accuracy_1 = 100 - rf_mape_1
rf_accuracy_7 = 100 - rf_mape_7
rf_accuracy_14 = 100 - rf_mape_14

In [ ]:
surge_threshold = df['target_1'].mean() + df['target_1'].std()


KeyError: 'target_1'

In [ ]:
import numpy as np

def surge_lead_time(predictions, actuals, threshold):

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    predicted_surge = np.where(predictions > threshold)[0]
    actual_surge = np.where(actuals > threshold)[0]

    lead_times = []

    for a in actual_surge:
        earlier = predicted_surge[predicted_surge < a]

        if len(earlier) > 0:
            lead_times.append(a - earlier.max())

    if len(lead_times) == 0:
        return 0

    return np.mean(lead_times)

In [ ]:
rf_lead_1 = surge_lead_time(predictions, actuals, surge_threshold)
rf_lead_7 = surge_lead_time(predictions, actuals, surge_threshold)
rf_lead_14 = surge_lead_time(predictions, actuals, surge_threshold)

In [ ]:
ets_lead_1 = surge_lead_time(predictions_ets_1, y_test, surge_threshold)
ets_lead_7 = surge_lead_time(predictions_ets_7, actual_7, surge_threshold)
ets_lead_14 = surge_lead_time(predictions_ets_14, actual_14, surge_threshold)

In [ ]:
arima_lead_1 = surge_lead_time(predictions_arima, y_test, surge_threshold)
arima_lead_7 = surge_lead_time(predictions_arima_7, actual_7, surge_threshold)
arima_lead_14 = surge_lead_time(predictions_arima_14, actual_14, surge_threshold)

In [ ]:
surge_table = pd.DataFrame({

"Model":["RF","ETS","ARIMA"],

"1-Day Lead Time":[rf_lead_1, ets_lead_1, arima_lead_1],

"7-Day Lead Time":[rf_lead_7, ets_lead_7, arima_lead_7],

"14-Day Lead Time":[rf_lead_14, ets_lead_14, arima_lead_14]

})

print(surge_table)

In [ ]:
print("Threshold:", surge_threshold)

print("Max actual demand:", df['target_1'].max())

Surge Lead Time Interpretation

The surge lead time for all models across horizons was observed to be zero days. This indicates that while the models were able to capture demand surges, they did not provide advance warning prior to the surge event. The forecasts reacted to spikes as they occurred rather than anticipating them ahead of time. This suggests that the models rely heavily on recent demand patterns and may require additional leading indicators to improve early surge detection capability.

In [ ]:
capacity_limit = df['target_1'].quantile(0.85)

In [ ]:
rf_breach_1 = np.mean(np.array(predictions) > capacity_limit)

rf_breach_7 = np.mean(np.array(predictions) > capacity_limit)

rf_breach_14 = np.mean(np.array(predictions) > capacity_limit)

In [ ]:
ets_breach_1 = np.mean(np.array(predictions_ets_1) > capacity_limit)

ets_breach_7 = np.mean(np.array(predictions_ets_7) > capacity_limit)

ets_breach_14 = np.mean(np.array(predictions_ets_14) > capacity_limit)

In [ ]:
arima_breach_1 = np.mean(np.array(predictions_arima) > capacity_limit)

arima_breach_7 = np.mean(np.array(predictions_arima_7) > capacity_limit)

arima_breach_14 = np.mean(np.array(predictions_arima_14) > capacity_limit)

In [ ]:
breach_table = pd.DataFrame({

"Model":["RF","ETS","ARIMA"],

"1-Day Breach Prob":[rf_breach_1, ets_breach_1, arima_breach_1],

"7-Day Breach Prob":[rf_breach_7, ets_breach_7, arima_breach_7],

"14-Day Breach Prob":[rf_breach_14, ets_breach_14, arima_breach_14]

})

print(breach_table)

The capacity breach probability for all models across forecasting horizons was observed to be zero. This indicates that none of the models predicted demand levels exceeding the defined capacity threshold (90th percentile of historical demand). While this suggests a low operational risk, it may also indicate that the models tend to smooth extreme spikes and may underestimate peak demand levels.

In [ ]:
import numpy as np

# -------- STABILITY INDEX --------

rf_stability = np.std([rf_mae_1, rf_mae_7, rf_mae_14])

ets_stability = np.std([mae_ets_1, mae_ets_7, mae_ets_14])

arima_stability = np.std([mae_arima_1, mae_arima_7, mae_arima_14])

In [ ]:
# -------- ROBUSTNESS --------

rf_robustness = rf_mae_14 / rf_mae_1

ets_robustness = mae_ets_14 / mae_ets_1

arima_robustness = mae_arima_14 / mae_arima_1

In [ ]:
kpi_table = pd.DataFrame({

"Model": ["Random Forest","ETS","ARIMA"],


"Stability Index":[rf_stability, ets_stability, arima_stability],

"Robustness":[rf_robustness, ets_robustness, arima_robustness],


})

print(kpi_table)

In [ ]:
kpi_table = pd.DataFrame({

"Model": ["Random Forest","Random Forest","Random Forest",
          "ETS","ETS","ETS",
          "ARIMA","ARIMA","ARIMA"],

"Horizon": ["1-Day","7-Day","14-Day",
            "1-Day","7-Day","14-Day",
            "1-Day","7-Day","14-Day"],

"Forecast Accuracy":[rf_accuracy_1, rf_accuracy_7, rf_accuracy_14,
                     ets_accuracy_1, ets_accuracy_7, ets_accuracy_14,
                     arima_accuracy_1, arima_accuracy_7, arima_accuracy_14],

"Breach Probability":[rf_breach_1, rf_breach_7, rf_breach_14,
                      ets_breach_1, ets_breach_7, ets_breach_14,
                      arima_breach_1, arima_breach_7, arima_breach_14],

"Surge Lead Time":[rf_lead_1, rf_lead_7, rf_lead_14,
                   ets_lead_1, ets_lead_7, ets_lead_14,
                   arima_lead_1, arima_lead_7, arima_lead_14]

})

print(kpi_table)

In [ ]:
kpi_table.to_csv("kpi_table.csv", index=False)